# LoanShield: MuRIL review classifier (GPU)

Fine-tunes `google/muril-base-cased` (fallback `ai4bharat/indic-bert`) at review level, using each review's app label, then averages review probabilities per app (PLAN.md 6.3).

**Uses the same developer-grouped folds as the local experiments**, so its scores can be fused and compared fairly.

**Upload to the Colab session** (Files panel):
- `data/processed/reviews_clean.parquet`
- `data/processed/folds_grouped.csv` (written by `python -m loanshield.models.run_experiments --exp main`)

**Output:** `muril_grouped.csv` (`app_id, score`). Save it in the repo as `results/oof/muril_ft_grouped.csv` (the name `muril_grouped.csv` is taken by the frozen-MuRIL run), then re-run `--exp main`. Fine-tuned MuRIL then appears in the comparison table and is eligible for fusion.

If no GPU is available, the notebook switches to **frozen MuRIL embeddings + logistic regression** (the plan's fallback). Log that in `docs/deviations.md`.

In [ ]:
!pip -q install transformers accelerate pyarrow scikit-learn

In [ ]:
import numpy as np, pandas as pd, torch, random, os
from torch import nn
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, AutoModel,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.linear_model import LogisticRegression

SEED, MAX_LEN, EPOCHS, LR, BATCH = 42, 128, 4, 2e-5, 16
MAX_REVIEWS_PER_APP = 40   # stops a few heavily reviewed apps dominating training
MODELS = ["google/muril-base-cased", "ai4bharat/indic-bert"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none -> frozen-embedding fallback")

In [ ]:
rv = pd.read_parquet("reviews_clean.parquet", columns=["app_id", "text"])
folds = pd.read_csv("folds_grouped.csv")          # app_id, fold, y
rv = rv.merge(folds, on="app_id", how="inner")
rv = rv.groupby("app_id", group_keys=False).apply(lambda g: g.sample(min(len(g), MAX_REVIEWS_PER_APP), random_state=SEED))
print(len(rv), "reviews,", rv.app_id.nunique(), "apps,", folds.fold.nunique(), "folds")
print(rv.groupby("y").app_id.nunique())

In [ ]:
def load_model(cls, **kw):
    for name in MODELS:
        try:
            return name, AutoTokenizer.from_pretrained(name), cls.from_pretrained(name, **kw)
        except Exception as e:
            print("could not load", name, "-", e)
    raise RuntimeError("no model could be loaded")

class Reviews(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tok):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.labels[i])
        return item

class WeightedTrainer(Trainer):
    """Cross-entropy with class weights (the unverified class is rare)."""
    def __init__(self, *a, class_weights=None, **kw):
        super().__init__(*a, **kw); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels, weight=self.cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss

In [ ]:
def finetune_fold(tr, te):
    name, tok, model = load_model(AutoModelForSequenceClassification, num_labels=2)
    pos = tr.y.mean()
    cw = torch.tensor([0.5 / (1 - pos), 0.5 / pos], dtype=torch.float)
    args = TrainingArguments(output_dir="out", num_train_epochs=EPOCHS, learning_rate=LR,
                             per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
                             warmup_ratio=0.1, weight_decay=0.01, max_grad_norm=1.0,
                             # fp16 made MuRIL collapse to a constant output in the first run
                             bf16=GPU and torch.cuda.is_bf16_supported(), seed=SEED, report_to=[], save_strategy="no", logging_steps=200)
    trainer = WeightedTrainer(model=model, args=args, train_dataset=Reviews(tr.text, tr.y, tok),
                              data_collator=DataCollatorWithPadding(tok), class_weights=cw)
    trainer.train()
    logits = trainer.predict(Reviews(te.text, te.y, tok)).predictions
    return name, torch.softmax(torch.tensor(logits), -1)[:, 1].numpy()

@torch.no_grad()
def embed(texts, tok, enc, bs=64):
    dev = "cuda" if GPU else "cpu"; enc.to(dev).eval(); out = []
    for i in range(0, len(texts), bs):
        b = tok(list(texts[i:i + bs]), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(dev)
        h = enc(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1)
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())    # mean pooling
    return np.vstack(out)

def frozen_fold(tr, te, cache={}):
    if "enc" not in cache:
        cache["name"], cache["tok"], cache["enc"] = load_model(AutoModel)
    Xtr = embed(tr.text.values, cache["tok"], cache["enc"]); Xte = embed(te.text.values, cache["tok"], cache["enc"])
    clf = LogisticRegression(class_weight="balanced", max_iter=3000, random_state=SEED).fit(Xtr, tr.y)
    return cache["name"] + " (frozen)", clf.predict_proba(Xte)[:, 1]

In [ ]:
scores = []
for k in sorted(folds.fold.unique()):
    tr, te = rv[rv.fold != k], rv[rv.fold == k]
    name, p = (finetune_fold if GPU else frozen_fold)(tr, te)
    app = pd.Series(p, index=te.index).groupby(te.app_id).mean()   # mean P(unverified) per app
    if app.std() < 1e-3:
        print(f"WARNING fold {k}: near-constant scores (std {app.std():.1e}) - training collapsed, re-run this fold")
    scores.append(app)
    print(f"fold {k}: {name}, {len(te)} reviews, {len(app)} apps")

out = pd.concat(scores).rename("score").rename_axis("app_id").reset_index()
out.to_csv("muril_ft_grouped.csv", index=False)
print(out.describe())

Download `muril_grouped.csv` and save it in the repo as `results/oof/muril_ft_grouped.csv`. Apps with no reviews are absent, and fusion falls back to the tabular score for them.

Record the model actually used (printed above) and whether it ran fine-tuned or frozen in `docs/deviations.md`.